In [1]:
import selenium.webdriver
from selenium.webdriver.common.by import By
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib import request
import bs4
import pandas as pd

In [2]:
http='https://fr.wikipedia.org/wiki/%C3%89lection_pr%C3%A9sidentielle_en_France#R%C3%A9sultats_depuis_1965'
request_text = request.urlopen(http).read()
page = bs4.BeautifulSoup(request_text, "lxml")
print(page.title)

<title>Élection présidentielle en France — Wikipédia</title>


In [3]:
table=page.find("table",attrs={'class':'wikitable'})


In [4]:
pres=[]
for elec in table.tbody.find_all('tr')[2:]:
    year=int(elec.find('td').text)
    
    if year>=2007:
        election=[]
        for info in elec.find_all('td'):
            election.append(info.text.strip())
        
        pres.append(election)


In [5]:
def separate(a):
    for i in range(len(a)):
        if a[i].isnumeric():
            return (a[:i].strip().lower(),float(a[i:].replace(',','.').strip()))

In [6]:
pres=[]
for elec in table.tbody.find_all('tr')[2:]:
    year=int(elec.find('td').text)
    
    if year>=2007:
        election=[]
        for info in elec.find_all('td'):
            try:
                election+=[a.strip() for a in info.text.split('%') if a.strip()]
            except:
                election.append(info.text)

        dico=[]#{'year':year}
        for i in election[1:-1]:
            try:
                a=i[i.index('('):].replace('(','').replace(')','')
                b=separate(a)
                dico.append([year,b[0],b[1]])
            except:
                continue
        pres+=dico

In [7]:
df=pd.DataFrame(pres,columns=['year','partis','resultat'])


In [ ]:
to_add={'year':[],'partis':[],'resultat':[]}
to_drop=[]
for i in range(len(df)):
    if 'soutien' in df.iloc[i]['partis']:
        partis=df.iloc[i]['partis'].split(', soutien ')
        res=[0.8*df.iloc[i]['resultat'],0.2*df.iloc[i]['resultat']]
        to_add['year'].append(df.iloc[i]['year'])
        to_add['partis'].append(partis[0])
        to_add['resultat'].append(res[0])
        to_add['year'].append(df.iloc[i]['year'])
        to_add['partis'].append(partis[1])
        to_add['resultat'].append(res[1])
        #first={'year':df.iloc[i]['year'],'partis':partis[0],'resultat':res[0]}
       # second={'year':df.iloc[i]['year'],'partis':partis[1],'resultat':res[1]}
       # to_add+=[first,second]
        #df.drop(index=i,inplace=True)
        to_drop.append(i)
for i in to_drop:
    df.drop(index=i,inplace=True)
df2=pd.DataFrame(to_add)
df=pd.concat([df,df2],ignore_index=True).reset_index()
df.drop(columns=['index'],inplace=True)


,index,year,partis,resultat
0,0,2007,lcr,4.080
1,1,2007,lo,1.330
2,2,2007,pt,0.340
3,3,2007,pcf,1.930
4,4,2007,ps,25.870
5,5,2007,les verts,1.570
6,6,2007,udf,18.570
7,7,2007,ump,31.180
8,8,2007,mpf,2.230
9,9,2007,fn,10.440


In [11]:
df.to_csv('pres.csv')